In [1]:
!pip install -q gradio tensorflow


In [5]:
from google.colab import files
uploaded = files.upload()

Saving class_names.json to class_names (1).json
Saving waste_background.png to waste_background.png
Saving waste_classifier.keras to waste_classifier (1).keras


In [6]:
import os
import json
import base64
import numpy as np
import tensorflow as tf
import gradio as gr

from PIL import Image
from io import BytesIO
from tensorflow.keras.utils import img_to_array

# ==========================================
# 1. LOAD MODEL AND FILES
# ==========================================

MODEL_PATH = "/content/waste_classifier.keras"
CLASS_PATH = "/content/class_names.json"
BG_PATH = "/content/waste_background.png"

for path in [MODEL_PATH, CLASS_PATH, BG_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"File not found: {path}\n"
            f"Available files: {os.listdir('/content')}"
        )

model = tf.keras.models.load_model(MODEL_PATH, compile=False)

with open(CLASS_PATH, "r") as f:
    class_names = json.load(f)

IMG_SIZE = (128, 128)

labels = {
    "O": "Organic Waste",
    "R": "Recyclable Waste"
}

# ==========================================
# 2. PREPARE BACKGROUND IMAGE
# ==========================================

bg_image = Image.open(BG_PATH).convert("RGB")
bg_image.thumbnail((1600, 1200))

buffer = BytesIO()
bg_image.save(buffer, format="JPEG", quality=85)

encoded_image = base64.b64encode(
    buffer.getvalue()
).decode("utf-8")

bg_url = "data:image/jpeg;base64," + encoded_image

print("Model and background image loaded successfully!")

# ==========================================
# 3. CLASSIFICATION FUNCTION
# ==========================================

def classify_waste(image):
    if image is None:
        return "Please upload a waste image.", {}

    image = image.convert("RGB").resize(IMG_SIZE)

    arr = img_to_array(image)
    arr = np.expand_dims(arr, axis=0)

    prediction = model.predict(arr, verbose=0)[0]

    scores = {
        labels[class_names[i]]: float(prediction[i])
        for i in range(len(class_names))
    }

    predicted_class = max(scores, key=scores.get)
    confidence = scores[predicted_class] * 100

    result = (
        f"Predicted Class: {predicted_class}\n"
        f"Confidence: {confidence:.2f}%"
    )

    return result, scores

# ==========================================
# 4. FULL-PAGE BACKGROUND CSS
# ==========================================

custom_css = """
/* Apply background to the complete browser page */
html,
body,
.gradio-container,
.gradio-container main,
gradio-app {
    background-color: #07151d !important;
}

/* Main Gradio application */
.gradio-container {
    max-width: 1150px !important;
    width: calc(100% - 36px) !important;
    margin: 20px auto !important;
    padding: 22px !important;
    background: rgba(5, 20, 28, 0.82) !important;
    border: 1px solid #00d6a0 !important;
    border-radius: 20px !important;
    box-sizing: border-box !important;
}

/* Header */
#main-title {
    text-align: center;
    color: #00e6a0;
    font-size: clamp(25px, 3vw, 36px);
    font-weight: 800;
}

#subtitle {
    text-align: center;
    color: white;
    font-size: 15px;
    margin-top: 10px;
}

/* Upload and result panels */
.panel {
    background: rgba(19, 37, 49, 0.94) !important;
    border: 1px solid rgba(0, 230, 160, 0.55) !important;
    border-radius: 14px !important;
    padding: 16px !important;
    box-sizing: border-box !important;
}

/* Panel headings */
.panel-title {
    color: white;
    background: linear-gradient(90deg, #5145e5, #06b68b);
    display: inline-block;
    padding: 9px 13px;
    border-radius: 7px;
    font-weight: bold;
    margin-bottom: 12px;
}

/* Classification button */
#predict-btn {
    background: linear-gradient(90deg, #5145e5, #06b68b) !important;
    color: white !important;
    border: none !important;
    border-radius: 10px !important;
    min-height: 48px;
    font-size: 16px !important;
    font-weight: bold !important;
}

#predict-btn:hover {
    filter: brightness(1.12);
}

/* Prediction box */
#result-box textarea {
    border: 1px solid #00d6a0 !important;
}

/* Footer */
#footer-text {
    text-align: center;
    color: #d1d5db;
    padding: 18px 5px 5px;
    margin-top: 18px;
    border-top: 1px solid rgba(0, 230, 160, 0.3);
}

footer {
    display: none !important;
}

/* Mobile layout */
@media (max-width: 700px) {
    .gradio-container {
        width: calc(100% - 16px) !important;
        margin: 8px auto !important;
        padding: 10px !important;
    }

    .panel {
        padding: 12px !important;
    }
}
"""

# Add the image as a CSS custom property.
# This lets CSS use the embedded image without a local file URL.

custom_css += """
html {
    --waste-page-background: url("BACKGROUND_IMAGE");
}
"""

custom_css = custom_css.replace(
    "BACKGROUND_IMAGE",
    bg_url
)

# Apply the image to the outer page and visible Gradio regions.
custom_css += """
html,
body,
body > gradio-app,
gradio-app {
    background-image:
        linear-gradient(
            rgba(2, 15, 20, 0.42),
            rgba(2, 15, 20, 0.55)
        ),
        var(--waste-page-background) !important;

    background-size: cover !important;
    background-position: center center !important;
    background-attachment: fixed !important;
    background-repeat: no-repeat !important;
}
"""

# ==========================================
# 5. BUILD INTERFACE
# ==========================================

with gr.Blocks(
    css=custom_css,
    theme=gr.themes.Soft()
) as demo:

    gr.HTML("""
        <div style="text-align:center; padding:12px 5px 22px;">
            <div id="main-title">
                ♻️ Smart Waste Classification System
            </div>
            <div id="subtitle">
                AI-powered waste detection for a cleaner and greener future
            </div>
            <div style="
                color:#6dffbd;
                font-weight:bold;
                font-size:13px;
                margin-top:12px;
            ">
                REDUCE &nbsp; • &nbsp; REUSE &nbsp; • &nbsp; RECYCLE
            </div>
        </div>
    """)

    with gr.Row(equal_height=True):

        with gr.Column(scale=1, min_width=280):
            with gr.Group(elem_classes=["panel"]):

                gr.HTML(
                    '<div class="panel-title">'
                    '📤 Waste Image Input</div>'
                )

                image_input = gr.Image(
                    type="pil",
                    label="Upload Waste Image",
                    height=280
                )

                predict_button = gr.Button(
                    "🔍 CLASSIFY WASTE",
                    elem_id="predict-btn"
                )

        with gr.Column(scale=1, min_width=280):
            with gr.Group(elem_classes=["panel"]):

                gr.HTML(
                    '<div class="panel-title">'
                    '🌱 Classification Results</div>'
                )

                result_output = gr.Textbox(
                    label="Prediction",
                    lines=3,
                    interactive=False,
                    elem_id="result-box"
                )

                probability_output = gr.Label(
                    label="Class Probabilities",
                    num_top_classes=2
                )

    gr.HTML("""
        <div id="footer-text">
            🌍 Smart Technology for a Cleaner Planet
            &nbsp; | &nbsp;
            Waste Classification Using Computer Vision
        </div>
    """)

    predict_button.click(
        fn=classify_waste,
        inputs=image_input,
        outputs=[result_output, probability_output]
    )

# ==========================================
# 6. LAUNCH APPLICATION
# ==========================================

demo.launch(share=True)

Model and background image loaded successfully!


/tmp/ipykernel_516/4184949260.py:233: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d767cd3fa91a54531c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
